In [2]:

# To develop 2 agents each of completely different sources of information and routing the user query to its agent and get the final answer

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model


#  Define State
class AgentState(TypedDict):
    query: str
    selected_agent: str
    answer: str


# Initialize LLM
llm = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0
)



E:\EDU_CARE\arg_venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
E:\EDU_CARE\arg_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# Supervisor Node > Decides which agent to use
def supervisor_node(state: AgentState) -> AgentState:

    query = state["query"]
    print("supervisor_node got query: ",query)

    prompt = f"""
    You are an Supervisor assistant.
    You need to choose either HR agent or IT agent.
    Select Correct agent only from the User question.
    Don't give explaination.

    Question: {query}
    """

    response = llm.invoke(prompt)

    state["selected_agent"] = response.content

    print("Selected agent is >>>>>>>>>>>>>>>>>> ",state["selected_agent"])

    return state


# HR Agent Node

def hr_agent_node(state: AgentState) -> AgentState:

    query = state["query"]

    prompt = f"""
    You are an HR support assistant.
    Answer the employee question clearly.

    Question: {query}
    """

    response = llm.invoke(prompt)

    state["answer"] = response.content

    return state


# IT Agent Node

def it_agent_node(state: AgentState) -> AgentState:

    query = state["query"]

    prompt = f"""
    You are an IT support assistant.
    Help fix the technical issue.

    Question: {query}
    """

    response = llm.invoke(prompt)

    state["answer"] = response.content

    return state


# Routing Function

def router(state: AgentState):

    if state["selected_agent"] == "hr_agent":
        return "hr_agent"

    else:
        return "it_agent"



In [4]:
# Build Graph

graph = StateGraph(AgentState)

graph.add_node("supervisor", supervisor_node)
graph.add_node("hr_agent", hr_agent_node)
graph.add_node("it_agent", it_agent_node)

graph.add_edge(START, "supervisor")

graph.add_conditional_edges(
    "supervisor",
    router,
    {
        "hr_agent": "hr_agent",
        "it_agent": "it_agent"
    }
)

graph.add_edge("hr_agent", END)
graph.add_edge("it_agent", END)

# Compile graph
app = graph.compile()

# Show the agent
from IPython.display import Image, display
display(Image(app.get_graph(xray=True).draw_mermaid_png()))

ValueError: Found edge starting at unknown node 'test_agent'

In [ ]:

# Example 1 — HR Query
result1 = app.invoke({
    "query": "How many leave days do I have?"
})

print("\nHR Agent Response:")
print(result1["answer"])



In [ ]:

# Example 2 — IT Query
result2 = app.invoke({
    "query": "My VPN is not working"
})

print("\nIT Agent Response:")
print(result2["answer"])
